In [1]:
import sys
!{sys.executable} -m pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://download.pytorch.org/whl/cpu
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
   ---------------------------------------- 0.0/122.9 MB ? eta -:--:--
   ---------------------------------------- 0.3/122.9 MB ? eta -:--:--
   ---------------------------------------- 0.3/122.9 MB ? eta -:--:--
   ---------------------------------------- 0.5/122.9 MB 621.2 kB/s eta 0:03:18
   ---------------------------------------- 0.8/122.9 MB 817.9 kB/s eta 0:02:30
   ---------------------------------------- 0.8/122.9 MB 817.9 kB/s eta 0:02:30
   ---------------------------------------- 1.0/122.9 MB 811.6 kB/s eta 0:02:31
   ---------------------------------------- 1.3/122.9 MB 817.9 kB/s eta 0:02:29
    --------------------------------------- 1.6/122.9 MB 931.6 kB/s eta 0:02:11
    --------------------------------------- 1.8/122.9 MB 958.5 kB/s eta 0:02:07
    --------------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [2]:
%matplotlib inline
import torch
print(torch.__version__)

2.12.0+cpu


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

# ── 1. Load data ──────────────────────────────────────────────
df = pd.read_csv('../data/raw/bandung_rainfall_2010_2026.csv', 
                 parse_dates=['date'], 
                 index_col='date')
X = df['precipitation_mm'].values  # shape: (5995,)

# ── 2. Train/test split (80/20, konsisten dengan Fase 2 & 3) ──
n = len(X)
split = int(n * 0.8)
X_train, X_test = X[:split], X[split:]

print(f"Total hari   : {n}")
print(f"Training set : {len(X_train)} hari")
print(f"Test set     : {len(X_test)} hari")

# ── 3. Z-score normalization (fit HANYA pada training set) ────
mu    = X_train.mean()
sigma = X_train.std()

X_train_scaled = (X_train - mu) / sigma
X_test_scaled  = (X_test  - mu) / sigma

print(f"\nTraining set setelah scaling:")
print(f"  mean : {X_train_scaled.mean():.4f}  (harusnya ~0)")
print(f"  std  : {X_train_scaled.std():.4f}   (harusnya ~1)")

# ── 4. Windowing function ─────────────────────────────────────
def create_windows(series, L):
    """
    Mengubah time series 1D menjadi pasangan (input, target).
    Input  : window panjang L  → shape (N, L)
    Target : nilai berikutnya  → shape (N,)
    """
    Xs, ys = [], []
    for i in range(len(series) - L):
        Xs.append(series[i : i + L])
        ys.append(series[i + L])
    return np.array(Xs), np.array(ys)

L = 60  # sequence length: 60 hari ke belakang

X_tr, y_tr = create_windows(X_train_scaled, L)
X_te, y_te = create_windows(X_test_scaled,  L)

print(f"\nShape setelah windowing:")
print(f"  X_tr : {X_tr.shape}  → (samples, sequence_length)")
print(f"  y_tr : {y_tr.shape}")
print(f"  X_te : {X_te.shape}")
print(f"  y_te : {y_te.shape}")

Total hari   : 5995
Training set : 4796 hari
Test set     : 1199 hari

Training set setelah scaling:
  mean : -0.0000  (harusnya ~0)
  std  : 1.0000   (harusnya ~1)

Shape setelah windowing:
  X_tr : (4736, 60)  → (samples, sequence_length)
  y_tr : (4736,)
  X_te : (1139, 60)
  y_te : (1139,)


In [4]:
# ── Step 3: LSTM Cell dari Scratch (numpy) ────────────────────

class LSTMCellNumpy:
    """
    LSTM cell satu timestep, diimplementasi dari persamaan matematis.
    Tidak ada autograd — hanya forward pass untuk demonstrasi.
    """
    def __init__(self, input_size, hidden_size, seed=42):
        rng   = np.random.default_rng(seed)
        scale = np.sqrt(2.0 / (input_size + hidden_size))  # Xavier init

        # Input weights  : shape (hidden_size, input_size)
        self.W_f = rng.normal(0, scale, (hidden_size, input_size))
        self.W_i = rng.normal(0, scale, (hidden_size, input_size))
        self.W_c = rng.normal(0, scale, (hidden_size, input_size))
        self.W_o = rng.normal(0, scale, (hidden_size, input_size))

        # Recurrent weights : shape (hidden_size, hidden_size)
        self.U_f = rng.normal(0, scale, (hidden_size, hidden_size))
        self.U_i = rng.normal(0, scale, (hidden_size, hidden_size))
        self.U_c = rng.normal(0, scale, (hidden_size, hidden_size))
        self.U_o = rng.normal(0, scale, (hidden_size, hidden_size))

        # Bias — forget gate diinisialisasi ke 1 (bukan 0)
        # Trik standar: b_f=1 → f_t ≈ σ(1) ≈ 0.73 di awal training
        # artinya network cenderung "ingat" dulu, baru belajar apa yang perlu dilupakan
        self.b_f = np.ones(hidden_size)
        self.b_i = np.zeros(hidden_size)
        self.b_c = np.zeros(hidden_size)
        self.b_o = np.zeros(hidden_size)

    @staticmethod
    def sigmoid(z):
        return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

    def forward_step(self, x_t, h_prev, c_prev):
        """
        Satu timestep.
        x_t    : shape (input_size,)
        h_prev : shape (hidden_size,)
        c_prev : shape (hidden_size,)
        """
        f_t     = self.sigmoid(self.W_f @ x_t + self.U_f @ h_prev + self.b_f)
        i_t     = self.sigmoid(self.W_i @ x_t + self.U_i @ h_prev + self.b_i)
        c_tilde = np.tanh(   self.W_c @ x_t + self.U_c @ h_prev + self.b_c)
        o_t     = self.sigmoid(self.W_o @ x_t + self.U_o @ h_prev + self.b_o)

        c_t = f_t * c_prev + i_t * c_tilde   # cell state update
        h_t = o_t * np.tanh(c_t)             # hidden state update

        return h_t, c_t, {'f': f_t, 'i': i_t, 'c_tilde': c_tilde, 'o': o_t}

    def forward_sequence(self, sequence):
        """
        Forward pass untuk satu window penuh.
        sequence : shape (L, input_size)
        """
        L, _      = sequence.shape
        d_h       = self.b_f.shape[0]
        h         = np.zeros(d_h)
        c         = np.zeros(d_h)
        gate_log  = []

        for t in range(L):
            h, c, gates = self.forward_step(sequence[t], h, c)
            gate_log.append(gates)

        return h, c, gate_log  # kembalikan state akhir


# ── Demonstrasi pada satu window dari data kita ───────────────
cell  = LSTMCellNumpy(input_size=1, hidden_size=32)

# Ambil window pertama dari training set, reshape ke (L, input_size)
sample_window = X_tr[0].reshape(-1, 1)   # shape (60, 1)
h_final, c_final, gate_log = cell.forward_sequence(sample_window)

# Inspeksi gate values di timestep terakhir
last_gates = gate_log[-1]
print("=== Gate values di timestep terakhir (sebelum training) ===")
print(f"Forget gate  (f_t) — mean: {last_gates['f'].mean():.4f}, "
      f"min: {last_gates['f'].min():.4f}, max: {last_gates['f'].max():.4f}")
print(f"Input gate   (i_t) — mean: {last_gates['i'].mean():.4f}, "
      f"min: {last_gates['i'].min():.4f}, max: {last_gates['i'].max():.4f}")
print(f"Output gate  (o_t) — mean: {last_gates['o'].mean():.4f}, "
      f"min: {last_gates['o'].min():.4f}, max: {last_gates['o'].max():.4f}")
print(f"\nh_final shape : {h_final.shape}")
print(f"c_final shape : {c_final.shape}")
print(f"\nForward pass sukses — gates dan state terhitung tanpa autograd.")

=== Gate values di timestep terakhir (sebelum training) ===
Forget gate  (f_t) — mean: 0.7180, min: 0.4949, max: 0.8941
Input gate   (i_t) — mean: 0.5243, min: 0.3691, max: 0.7147
Output gate  (o_t) — mean: 0.5282, min: 0.3197, max: 0.7561

h_final shape : (32,)
c_final shape : (32,)

Forward pass sukses — gates dan state terhitung tanpa autograd.


In [5]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

X_tr_t = torch.FloatTensor(X_tr).unsqueeze(-1)
y_tr_t = torch.FloatTensor(y_tr)
X_te_t = torch.FloatTensor(X_te).unsqueeze(-1)
y_te_t = torch.FloatTensor(y_te)

train_loader = DataLoader(
    TensorDataset(X_tr_t, y_tr_t),
    batch_size=32, shuffle=True
)

class LSTMForecaster(nn.Module):
    def __init__(self, hidden_size=32):
        super().__init__()
        self.lstm         = nn.LSTM(input_size=1, hidden_size=hidden_size,
                                    num_layers=1, batch_first=True)
        self.output_layer = nn.Linear(hidden_size, 1)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        last_h      = lstm_out[:, -1, :]
        return self.output_layer(last_h).squeeze(-1)

torch.manual_seed(42)
model     = LSTMForecaster(hidden_size=32)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

n_epochs     = 200
train_losses = []

print(f"Total parameter: {sum(p.numel() for p in model.parameters())}\n")

for epoch in range(n_epochs):
    model.train()
    epoch_loss = 0.0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        y_pred = model(X_batch)
        loss   = criterion(y_pred, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        epoch_loss += loss.item() * len(X_batch)
    train_losses.append(epoch_loss / len(X_tr_t))
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:3d}/{n_epochs} | Loss: {train_losses[-1]:.5f}")

print("\nTraining selesai.")

Total parameter: 4513

Epoch  20/200 | Loss: 0.70667
Epoch  40/200 | Loss: 0.69862
Epoch  60/200 | Loss: 0.68035
Epoch  80/200 | Loss: 0.64909
Epoch 100/200 | Loss: 0.59721
Epoch 120/200 | Loss: 0.56228
Epoch 140/200 | Loss: 0.52140
Epoch 160/200 | Loss: 0.47897
Epoch 180/200 | Loss: 0.44975
Epoch 200/200 | Loss: 0.42701

Training selesai.


In [6]:
torch.save(model.state_dict(), '../outputs/lstm_model.pt')
print("Model tersimpan.")

Model tersimpan.


In [7]:
 # ── Step 5: Walk-forward validation ───────────────────────────
model.eval()

preds_scaled = []

with torch.no_grad():
    for i in range(len(X_te_t)):
        x     = X_te_t[i].unsqueeze(0)   # (1, 60, 1)
        y_hat = model(x).item()
        preds_scaled.append(y_hat)

preds_scaled = np.array(preds_scaled)

# Inverse transform ke satuan mm asli
preds_mm = preds_scaled * sigma + mu
y_te_mm  = y_te         * sigma + mu   # y_te masih scaled, inverse juga

# ── Metrik ────────────────────────────────────────────────────
mae  = np.mean(np.abs(preds_mm - y_te_mm))
rmse = np.sqrt(np.mean((preds_mm - y_te_mm)**2))

# Skill score vs naive (konsisten dengan Fase 2 & 3)
naive_mae  = np.mean(np.abs(np.diff(X_test)))   # |X_t - X_{t-1}|
naive_rmse = np.sqrt(np.mean(np.diff(X_test)**2))
skill_mae  = 1 - mae  / naive_mae
skill_rmse = 1 - rmse / naive_rmse

print("=== LSTM Walk-forward Validation ===")
print(f"MAE        : {mae:.4f} mm")
print(f"RMSE       : {rmse:.4f} mm")
print(f"Skill (MAE): {skill_mae:.4f}")
print(f"Skill(RMSE): {skill_rmse:.4f}")

print("\n=== Komparasi semua model ===")
print(f"{'Model':<20} {'MAE':>8} {'RMSE':>8}")
print("-" * 38)
print(f"{'Naive':<20} {'7.05':>8} {'8.74':>8}")
print(f"{'AR(2)':<20} {'5.50':>8} {'7.77':>8}")
print(f"{'ARMA(2,1)':<20} {'5.08':>8} {'7.65':>8}")
print(f"{'Kalman Filter':<20} {'4.93':>8} {'8.03':>8}")
print(f"{'LSTM':<20} {mae:>8.2f} {rmse:>8.2f}")

=== LSTM Walk-forward Validation ===
MAE        : 6.4263 mm
RMSE       : 9.7582 mm
Skill (MAE): -0.2062
Skill(RMSE): -0.0616

=== Komparasi semua model ===
Model                     MAE     RMSE
--------------------------------------
Naive                    7.05     8.74
AR(2)                    5.50     7.77
ARMA(2,1)                5.08     7.65
Kalman Filter            4.93     8.03
LSTM                     6.43     9.76


In [8]:
# Naive yang sejajar dengan walk-forward validation
naive_preds_mm  = X_test[59:59 + len(y_te_mm)]   # X_{t-1}
naive_targets_mm = X_test[60:60 + len(y_te_mm)]   # X_t

naive_mae  = np.mean(np.abs(naive_targets_mm - naive_preds_mm))
naive_rmse = np.sqrt(np.mean((naive_targets_mm - naive_preds_mm)**2))

skill_mae  = 1 - mae  / naive_mae
skill_rmse = 1 - rmse / naive_rmse

print(f"Naive MAE  (corrected): {naive_mae:.4f} mm")
print(f"Naive RMSE (corrected): {naive_rmse:.4f} mm")
print(f"Skill (MAE) : {skill_mae:.4f}")
print(f"Skill (RMSE): {skill_rmse:.4f}")

Naive MAE  (corrected): 5.3119 mm
Naive RMSE (corrected): 9.1890 mm
Skill (MAE) : -0.2098
Skill (RMSE): -0.0619


In [ ]:
# ── Training ulang dengan early stopping ─────────────────────
# Validation split dari training set (bukan test set)
val_size   = int(len(X_tr) * 0.1)
X_tr2, X_val = X_tr[:-val_size], X_tr[-val_size:]
y_tr2, y_val = y_tr[:-val_size], y_tr[-val_size:]

X_tr2_t = torch.FloatTensor(X_tr2).unsqueeze(-1)
y_tr2_t = torch.FloatTensor(y_tr2)
X_val_t = torch.FloatTensor(X_val).unsqueeze(-1)
y_val_t = torch.FloatTensor(y_val)

train_loader2 = DataLoader(
    TensorDataset(X_tr2_t, y_tr2_t),
    batch_size=32, shuffle=True
)

# Model baru
torch.manual_seed(42)
model2     = LSTMForecaster(hidden_size=32)
optimizer2 = torch.optim.Adam(model2.parameters(), lr=1e-3)

# Early stopping config
patience        = 30
best_val_loss   = float('inf')
best_state      = None
epochs_no_improve = 0
train_losses2   = []
val_losses2     = []
n_epochs2       = 1000

print(f"Train samples : {len(X_tr2)}")
print(f"Val samples   : {len(X_val)}")
print(f"Max epochs    : {n_epochs2}, patience: {patience}\n")

for epoch in range(n_epochs2):
    # ── Train ──
    model2.train()
    epoch_loss = 0.0
    for X_batch, y_batch in train_loader2:
        optimizer2.zero_grad()
        y_pred     = model2(X_batch)
        loss       = criterion(y_pred, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model2.parameters(), max_norm=1.0)
        optimizer2.step()
        epoch_loss += loss.item() * len(X_batch)
    train_losses2.append(epoch_loss / len(X_tr2_t))

    # ── Validation ──
    model2.eval()
    with torch.no_grad():
        val_pred = model2(X_val_t)
        val_loss = criterion(val_pred, y_val_t).item()
    val_losses2.append(val_loss)

    # ── Early stopping ──
    if val_loss < best_val_loss:
        best_val_loss     = val_loss
        best_state        = {k: v.clone() for k, v in model2.state_dict().items()}
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1:4d} | Train: {train_losses2[-1]:.5f} "
              f"| Val: {val_loss:.5f} | Best val: {best_val_loss:.5f} "
              f"| No improve: {epochs_no_improve}")

    if epochs_no_improve >= patience:
        print(f"\nEarly stopping di epoch {epoch+1}.")
        break

# Load best model
model2.load_state_dict(best_state)
torch.save(best_state, '../outputs/lstm_model_best.pt')
print(f"\nBest validation loss: {best_val_loss:.5f}")
print("Model terbaik tersimpan.")